In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuration
SILVER_SCHEMA = "workspace.silver"
GOLD_SCHEMA = "workspace.gold"

# Read silver layer tables
fact_transactions = spark.table(f"{SILVER_SCHEMA}.fact_transactions")
dim_customers = spark.table(f"{SILVER_SCHEMA}.dim_customers")
dim_products = spark.table(f"{SILVER_SCHEMA}.dim_products")
dim_stores = spark.table(f"{SILVER_SCHEMA}.dim_stores")

print("✓ Silver layer tables loaded successfully")

In [0]:
# Create daily sales aggregation
gold_daily_sales = (
    fact_transactions
    .join(dim_stores, "store_id")
    .join(dim_products, "product_id")
    .groupBy(
        "transaction_date",
        "store_id",
        "store_name",
        "store_city",
        "store_province",
        "category",
        "payment_method"
    )
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.sum("subtotal").alias("gross_sales"),
        F.sum("discount_amount").alias("total_discounts"),
        F.sum("total_amount").alias("net_sales"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.countDistinct("customer_id").alias("unique_customers")
    )
    .withColumn("discount_rate", 
                F.round(F.col("total_discounts") / F.col("gross_sales") * 100, 2))
    .withColumn("created_at", F.current_timestamp())
)

print(f"Daily sales records: {gold_daily_sales.count():,}")
display(gold_daily_sales.orderBy(F.desc("transaction_date")).limit(10))

In [0]:
# Create monthly sales aggregation
gold_monthly_sales = (
    fact_transactions
    .join(dim_stores, "store_id")
    .join(dim_products, "product_id")
    .withColumn("year", F.year("transaction_date"))
    .withColumn("month", F.month("transaction_date"))
    .withColumn("year_month", F.date_format("transaction_date", "yyyy-MM"))
    .groupBy(
        "year",
        "month",
        "year_month",
        "store_id",
        "store_name",
        "category"
    )
    .agg(
        F.count("transaction_id").alias("transaction_count"),
        F.sum("quantity").alias("total_quantity_sold"),
        F.sum("total_amount").alias("total_revenue"),
        F.sum("discount_amount").alias("total_discounts"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("transaction_date").alias("active_days")
    )
    .withColumn("avg_daily_revenue", 
                F.round(F.col("total_revenue") / F.col("active_days"), 2))
    .withColumn("created_at", F.current_timestamp())
)

print(f"Monthly sales records: {gold_monthly_sales.count():,}")
display(gold_monthly_sales.orderBy(F.desc("year_month")).limit(10))

In [0]:
# Store performance metrics
gold_store_performance = (
    fact_transactions
    .join(dim_stores, "store_id")
    .join(dim_products, "product_id")
    .groupBy(
        "store_id",
        "store_name",
        "store_city",
        "store_province",
        "store_type"
    )
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("total_amount").alias("total_revenue"),
        F.avg("total_amount").alias("avg_transaction_value"),
        F.sum("quantity").alias("total_units_sold"),
        F.countDistinct("customer_id").alias("unique_customers"),
        F.countDistinct("product_id").alias("unique_products_sold"),
        F.min("transaction_date").alias("first_transaction_date"),
        F.max("transaction_date").alias("last_transaction_date")
    )
    .withColumn("revenue_per_customer", 
                F.round(F.col("total_revenue") / F.col("unique_customers"), 2))
    .withColumn("created_at", F.current_timestamp())
)

# Add revenue rank
window_spec = Window.orderBy(F.desc("total_revenue"))
gold_store_performance = gold_store_performance.withColumn(
    "revenue_rank",
    F.row_number().over(window_spec)
)

print(f"Store performance records: {gold_store_performance.count():,}")
display(gold_store_performance.orderBy("revenue_rank").limit(10))

In [0]:
# Category sales trends over time
gold_category_trends = (
    fact_transactions
    .join(dim_products, "product_id")
    .withColumn("year_month", F.date_format("transaction_date", "yyyy-MM"))
    .groupBy(
        "year_month",
        "category",
        "subcategory"
    )
    .agg(
        F.sum("total_amount").alias("revenue"),
        F.sum("quantity").alias("units_sold"),
        F.count("transaction_id").alias("transaction_count"),
        F.avg("total_amount").alias("avg_transaction_value")
    )
    .withColumn("created_at", F.current_timestamp())
)

# Calculate month-over-month growth
window_spec = Window.partitionBy("category", "subcategory").orderBy("year_month")
gold_category_trends = (
    gold_category_trends
    .withColumn("prev_month_revenue", F.lag("revenue").over(window_spec))
    .withColumn(
        "mom_growth_pct",
        F.when(
            F.col("prev_month_revenue").isNotNull(),
            F.round((F.col("revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue") * 100, 2)
        )
    )
    .drop("prev_month_revenue")
)

print(f"Category trend records: {gold_category_trends.count():,}")
display(gold_category_trends.orderBy(F.desc("year_month"), F.desc("revenue")).limit(10))

In [0]:
# Write all gold tables
gold_daily_sales.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.daily_sales_summary")
print("✓ Wrote daily_sales_summary")

gold_monthly_sales.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.monthly_sales_summary")
print("✓ Wrote monthly_sales_summary")

gold_store_performance.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.store_performance")
print("✓ Wrote store_performance")

gold_category_trends.write.mode("overwrite").saveAsTable(f"{GOLD_SCHEMA}.category_trends")
print("✓ Wrote category_trends")

print("\n" + "="*50)
print("Sales Analytics Gold Layer - COMPLETED")
print("="*50)